# scikit-learn for Machine Learning — S01–S05
An end-to-end, trusted reproducible notebook. Predict before running; guided edits belong in a private local copy and never award platform mastery. All data are original fictional teaching observations. No uploads, credentials, Drive, paid service or remote dataset. Browser fixtures are not live training; this notebook actually fits the declared models. Genuine Colab remains UNVERIFIED.

In [ ]:
import os, sys, json, time, resource
os.environ.update({"OMP_NUM_THREADS":"1","OPENBLAS_NUM_THREADS":"1","MKL_NUM_THREADS":"1"})
started = time.monotonic()
import numpy as np
import pandas as pd
import matplotlib
import sklearn
assert sys.version_info[:3] == (3,13,16)
assert (np.__version__,pd.__version__,matplotlib.__version__,sklearn.__version__) == ("2.5.4","3.0.6","3.11.2","1.9.1")
results = {}
def dtype_name(dtype):
    return f"string[{dtype.storage}]" if isinstance(dtype, pd.StringDtype) else str(dtype)

def normalize_table(value):
    if isinstance(value, pd.DataFrame):
        assert value.shape[0] <= 32 and value.shape[1] <= 8
        return {"kind": "frame", "shape": list(value.shape), "index": normalize_table(value.index.tolist()), "index_names": normalize_table(value.index.names), "columns": normalize_table(value.columns.tolist()), "dtypes": [dtype_name(dtype) for dtype in value.dtypes], "rows": normalize_table(value.values.tolist())}
    if isinstance(value, pd.Series):
        assert len(value) <= 32
        return {"kind": "series", "name": normalize_table(value.name), "index": normalize_table(value.index.tolist()), "index_names": normalize_table(value.index.names), "dtype": dtype_name(value.dtype), "values": normalize_table(value.tolist())}
    if value is pd.NA or value is pd.NaT:
        return {"tag": "missing"}
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.generic):
        return normalize_table(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return {"tag": "missing"} if np.isnan(value) else {"tag": "nonfinite", "value": "+inf" if value > 0 else "-inf"}
    if isinstance(value, dict):
        return {key: normalize_table(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [normalize_table(item) for item in value]
    if isinstance(value, str):
        assert len(value) <= 120
    if isinstance(value, (float, int)) and not isinstance(value, bool):
        assert abs(value) <= 1e12
    return value


def normalize(value):
    if isinstance(value,np.ndarray): return normalize(value.tolist())
    if isinstance(value,dict): return {key:normalize(item) for key,item in value.items()}
    if isinstance(value,(list,tuple)): return [normalize(item) for item in value]
    return normalize_table(value)
def equivalent(actual,expected):
    if isinstance(expected,bool): return type(actual) is bool and actual == expected
    if isinstance(expected,(int,float)): return not isinstance(actual,bool) and isinstance(actual,(int,float)) and abs(actual-expected) <= 1e-9+1e-9*abs(expected)
    if isinstance(expected,list): return isinstance(actual,list) and len(actual)==len(expected) and all(equivalent(left,right) for left,right in zip(actual,expected))
    if isinstance(expected,dict): return isinstance(actual,dict) and actual.keys()==expected.keys() and all(equivalent(actual[key],expected[key]) for key in expected)
    return actual == expected


## Bundled data and provenance
The full original fictional16-house source is embedded here with permission, units, schema, split IDs and a verified source hash. Workflow snippets use only its declared id/area/age/neighborhood/price projection; target and future-derived fields are never input features. No external file or download is needed.

In [ ]:
import hashlib
canonical_houses = json.loads("{\"id\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\",\"H13\",\"H14\",\"H15\",\"H16\"],\"area\":[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16],\"age\":[0,5,null,10,0,5,null,10,0,5,null,10,0,5,null,10],\"neighborhood\":[\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\"],\"price\":[100,120,140,160,180,200,220,240,260,280,300,320,340,360,380,400],\"future_price\":[110,130,150,170,190,210,230,250,270,290,310,330,350,370,390,410],\"listed_on\":[\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\"]}")
dataset_manifest = json.loads("{\"id\":\"sklearn-houses.v1\",\"sha256\":\"751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5\",\"source\":\"original fictional canonical housing source reused without mutation\",\"permission\":\"platform teaching use\",\"train_ids\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\"],\"test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"limitations\":\"fictional teaching data; not population, causal or deployment evidence\",\"units\":{\"area\":\"area units\",\"age\":\"years\",\"price\":\"$k\",\"future_price\":\"quarantined $k\"},\"schema\":{\"id\":\"unique string\",\"area\":\"numeric\",\"age\":\"nullable numeric\",\"neighborhood\":\"category label\",\"price\":\"target\",\"future_price\":\"unavailable at prediction\",\"listed_on\":\"ISO date\"},\"seed\":42}")
assert hashlib.sha256(json.dumps(canonical_houses,sort_keys=True).encode()).hexdigest() == dataset_manifest["sha256"]
assert canonical_houses["id"][:12] == dataset_manifest["train_ids"]
assert canonical_houses["id"][12:] == dataset_manifest["test_ids"]
print(json.dumps(dataset_manifest,sort_keys=True))

## S01 — The estimator lifecycle and a first baseline
Prerequisites/recall: P05, T01–T05; Regression overview.

Estimators and transformers have learned state and different output contracts.

A model maps features to predictions; a transformer maps features to features.

Terminology: estimator, constructor, fit, predict, transform, fitted attribute, baseline.

An estimator has a lifecycle, not just a method name. Construction records choices such as strategy='mean'; fit learns state from permitted training examples. A mean regressor learns the training target average. A transformer instead learns a feature-coordinate rule such as a training mean and scale.

For training prices100,200,300, the mean baseline is(100+200+300)/3=200 ($k), so two new houses receive[200,200]. New feature rows determine output length, not the learned target mean. A constructor cannot invent that mean: predict before fit raises a named NotFittedError. Fitting on the evaluation targets would answer an easier, invalid question.

Predict maps a feature row to a model output; transform maps features to prepared features. Transform uses saved training state without consulting new target labels or refitting on the new batch. Connect this to Algebra mappings, P05 functions, NumPy shapes and the training-only rules from Data Preparation. The browser displays trusted frozen outcomes; a real notebook execution requires a fresh-kernel environment and output record.

Progressive experiment: Trace constructor/fit/predict, then a separate transform-without-refit case.

### S01.O1 — critical decision AND reason: Respect the unfitted/fitted lifecycle and training/evaluation boundary.
Predict first: Repair predict-before-fit or test-fit ordering and explain learned state.

Construction stores configuration, not the learned target mean. Fit learns from training features/targets; predict uses saved state on new features without learning their targets. An unfitted prediction raises NotFittedError, and fitting on test targets invalidates unseen evaluation.

Misconception: Constructing the estimator learns the mean, so predicting before fit or fitting on test targets is valid.

Remediation: Follow constructor→fit(train)→predict(new).

Guided change: Trace constructor/fit/predict, then a separate transform-without-refit case. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
train = np.array([[1.],[2.],[3.]])
target = np.array([100.,200.,300.])
new = np.array([[4.],[5.]])
model = DummyRegressor(strategy='mean')
try:
    model.predict(new)
except NotFittedError:
    before = {'tag':'error','class':'NotFittedError'}
model.fit(train,target)
result = {'before_fit':before,'training_mean':model.constant_,'new_predictions':model.predict(new),'fit_ids':['H01','H02','H03'],'new_ids':['H04','H05'],'test_target_fit_allowed':False}
expected = json.loads("{\"before_fit\":{\"tag\":\"error\",\"class\":\"NotFittedError\"},\"training_mean\":[[200]],\"new_predictions\":[200,200],\"fit_ids\":[\"H01\",\"H02\",\"H03\"],\"new_ids\":[\"H04\",\"H05\"],\"test_target_fit_allowed\":false}")
assert equivalent(normalize(result),expected)
results["S01.O1"] = normalize(result)
print(json.dumps({"objective":"S01.O1","output":normalize(result)},sort_keys=True))

### S01.O2: Predict a DummyRegressor training-mean baseline.
Predict first: Select new baseline predictions and output length.

A mean baseline ignores feature values but learns its constant from training targets only. Its prediction vector has one entry per new row, never a mean computed from test targets.

Misconception: Use the mean of evaluation targets as the baseline prediction because it gives the fairest test error.

Remediation: Compute the training target mean, not test mean.

Guided change: Trace constructor/fit/predict, then a separate transform-without-refit case. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
train = np.array([[1.],[2.],[3.]])
target = np.array([100.,200.,300.])
new = np.array([[4.],[5.]])
model = DummyRegressor(strategy='mean').fit(train,target)
new = np.zeros((2,1))
result = {'training_mean':model.constant_,'predictions':model.predict(new),'length':len(new),'source':'training targets only'}
expected = json.loads("{\"training_mean\":[[200]],\"predictions\":[200,200],\"length\":2,\"source\":\"training targets only\"}")
assert equivalent(normalize(result),expected)
results["S01.O2"] = normalize(result)
print(json.dumps({"objective":"S01.O2","output":normalize(result)},sort_keys=True))

### S01.O3 — critical decision AND reason: Apply transform to new features without refitting.
Predict first: Reject fit_transform on held-out data and justify transform.

Transform maps new features into saved training coordinates; predict maps features to model outputs. Neither inference step may refit on evaluation rows. A different test batch cannot change the training mean2.

Misconception: Predict and transform are interchangeable, and fit_transform should relearn coordinates on every test batch.

Remediation: Separate feature-state learning from prediction.

Guided change: Trace constructor/fit/predict, then a separate transform-without-refit case. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
scaler = StandardScaler().fit([[1.],[3.]])
new = [[5.],[103.]]
saved = scaler.mean_.copy()
transformed = scaler.transform(new)
result = {'saved_training_mean':saved,'after_transform_mean':scaler.mean_,'transformed_features':transformed,'predict_is_transform':False,'evaluation_refit_allowed':False}
expected = json.loads("{\"saved_training_mean\":[2],\"after_transform_mean\":[2],\"transformed_features\":[[3],[101]],\"predict_is_transform\":false,\"evaluation_refit_allowed\":false}")
assert equivalent(normalize(result),expected)
results["S01.O3"] = normalize(result)
print(json.dumps({"objective":"S01.O3","output":normalize(result)},sort_keys=True))

### S01.O4: Distinguish browser fixture simulation from real notebook execution.
Predict first: Choose what evidence establishes an actual fitted estimator.

These outputs come from trusted author execution. Displaying a frozen browser trace is not browser fitting or evidence that your edited notebook ran; an actual fresh-kernel run needs its own environment, fitted-state and assertion record.

Misconception: A browser animation or a downloaded notebook proves that real Python executed and fitted your estimator.

Remediation: Inspect the notebook execution manifest and fitted attributes.

Guided change: Trace constructor/fit/predict, then a separate transform-without-refit case. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
train = np.array([[1.],[2.],[3.]])
target = np.array([100.,200.,300.])
new = np.array([[4.],[5.]])
model = LinearRegression().fit(train,target)
result = {'trusted_author_fit':hasattr(model,'coef_'),'coefficients':model.coef_,'predictions':model.predict(new),'browser_execution':False,'required_notebook_evidence':'fresh-kernel manifest, fitted attributes and complete output assertions'}
expected = json.loads("{\"trusted_author_fit\":true,\"coefficients\":[99.99999999999997],\"predictions\":[399.99999999999994,499.99999999999994],\"browser_execution\":false,\"required_notebook_evidence\":\"fresh-kernel manifest, fitted attributes and complete output assertions\"}")
assert equivalent(normalize(result),expected)
results["S01.O4"] = normalize(result)
print(json.dumps({"objective":"S01.O4","output":normalize(result)},sort_keys=True))

## S02 — Evaluate against a meaningful baseline
Prerequisites/recall: S01; Regression metrics, Statistics imbalance.

A metric answers a specific evaluation question relative to a fair baseline.

Error units and denominators; majority-class accuracy can be misleading.

Terminology: MAE, MSE, RMSE, accuracy, class balance, DummyClassifier, baseline.

A baseline asks whether the workflow improves on a simple prediction rule under the same information and evaluation conditions. Fit both candidates only on the permitted training data, then compare their predictions against the same held-out IDs and targets. A model's training error and a baseline's test error answer different questions.

Write residual e_i=prediction_i−target_i for each of n evaluation rows. MAE=Σ|e_i|/n, MSE=Σe_i²/n and RMSE=√MSE. With targets100,200 and predictions120,180, residuals20,−20 give MAE20 ($k), MSE400 (($k)²), RMSE20 ($k). MSE emphasizes larger errors and changes units; do not compare its numeric magnitude directly with MAE. Read the requested metric before selecting its value and units.

Now consider a fictional Spam/Churn triage baseline: class1 is the positive case requiring detection, class0 is the majority. A training-fitted most-frequent classifier can predict0 for every record. On nine negatives and one positive it is90% accurate while detecting0 of1 positives. Accuracy counts all correct rows, not positive-case usefulness. Inspect actual positive counts before the combined critical check. Default regression score is R², classification score here is accuracy; neither a high score nor visual association establishes causation, calibration or universal generalization.

Progressive experiment: Compute regression units, then inspect a majority-class baseline's missed positives.

### S02.O1 — critical decision AND reason: Compare a model against a training-fitted baseline on the same held-out rows.
Predict first: Reject an unequal-split baseline comparison and explain the fairness boundary.

Both candidates learn from the same permitted training information and are evaluated against the same held-out targets in the same ID order. A training score or a baseline on easier/different rows is not a fair unseen-data comparison.

Misconception: Compare the model's training score with a baseline on different test rows; the smaller number proves improvement.

Remediation: Align row IDs and training information for both predictions.

Guided change: Compute regression units, then inspect a majority-class baseline's missed positives. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
train = np.array([[1.],[2.],[3.]])
target = np.array([100.,200.,300.])
new = np.array([[4.],[5.]])
baseline = DummyRegressor(strategy='mean').fit(train,target); model = LinearRegression().fit(train,target)
truth = np.array([400.,500.]); ids = ['H34','H35']
result = {'shared_held_out_ids':ids,'baseline_predictions':baseline.predict(new),'model_predictions':model.predict(new),'baseline_mae':mean_absolute_error(truth,baseline.predict(new)),'model_mae':mean_absolute_error(truth,model.predict(new)),'unequal_split_comparison_allowed':False,'training_score_is_test':False}
expected = json.loads("{\"shared_held_out_ids\":[\"H34\",\"H35\"],\"baseline_predictions\":[200,200],\"model_predictions\":[399.99999999999994,499.99999999999994],\"baseline_mae\":250,\"model_mae\":5.684341886080802e-14,\"unequal_split_comparison_allowed\":false,\"training_score_is_test\":false}")
assert equivalent(normalize(result),expected)
results["S02.O1"] = normalize(result)
print(json.dumps({"objective":"S02.O1","output":normalize(result)},sort_keys=True))

### S02.O2 — critical decision AND reason: Choose and calculate an appropriate metric with units/denominator and reject majority accuracy as proof of positive-case detection.
Predict first: Every form requires one regression metric/units decision plus an imbalance claim judged from the positive-case count table; the reason must justify the interpretation, not only name a metric.

MAE and RMSE have target units $k; MSE has squared units ($k)^2. Divide regression error sums by the two evaluated rows. The training-most-frequent classifier predicts zero, so its accuracy can be high while detecting none of the supplied positive cases. Both the metric calculation and count-based interpretation are required.

Misconception: MSE is measured in $k, a different denominator is harmless, and high majority accuracy proves useful positive-case detection.

Remediation: Rebuild the per-row error/count tables in two staged examples before combining the decisions.

Guided change: Compute regression units, then inspect a majority-class baseline's missed positives. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
truth = np.array([100.,200.]); prediction = np.array([120.,180.])
classifier = DummyClassifier(strategy='most_frequent').fit(np.zeros((10,1)),[0]*9+[1])
class_truth = np.array([0]*9+[1]*1); class_prediction = classifier.predict(np.zeros((len(class_truth),1)))
result = {'requested_metric':'mae','regression_rows':len(truth),'errors':prediction-truth,'mae':mean_absolute_error(truth,prediction),'mse':mean_squared_error(truth,prediction),'rmse':root_mean_squared_error(truth,prediction),'units':{'mae':'$k','mse':'($k)^2','rmse':'$k'},'negative_cases':9,'positive_cases':1,'positive_detected':int(((class_prediction==1)&(class_truth==1)).sum()),'accuracy':accuracy_score(class_truth,class_prediction),'majority_accuracy_proves_detection':False}
expected = json.loads("{\"requested_metric\":\"mae\",\"regression_rows\":2,\"errors\":[20,-20],\"mae\":20,\"mse\":400,\"rmse\":20,\"units\":{\"mae\":\"$k\",\"mse\":\"($k)^2\",\"rmse\":\"$k\"},\"negative_cases\":9,\"positive_cases\":1,\"positive_detected\":0,\"accuracy\":0.9,\"majority_accuracy_proves_detection\":false}")
assert equivalent(normalize(result),expected)
results["S02.O2"] = normalize(result)
print(json.dumps({"objective":"S02.O2","output":normalize(result)},sort_keys=True))

### S02.O3: Interpret default score cautiously and name the explicit metric.
Predict first: Choose R² versus classification accuracy meaning in a supplied API trace.

Default score depends on the estimator: LinearRegression reports R2, while this classifier reports accuracy. Name the actual metric and its evaluation rows instead of assuming every score is accuracy or a physical error.

Misconception: Every sklearn .score is accuracy and directly comparable across regression and classification.

Remediation: Read the estimator task and metric definition.

Guided change: Compute regression units, then inspect a majority-class baseline's missed positives. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
train = np.array([[1.],[2.],[3.]])
target = np.array([100.,200.,300.])
new = np.array([[4.],[5.]])
regressor = LinearRegression().fit(train,target)
classifier = DummyClassifier(strategy='most_frequent').fit(train,[0,0,1])
result = {'regression_score_name':'R2','regression_score':regressor.score(new,[420.,480.]),'classification_score_name':'accuracy','classification_score':classifier.score(new,[0,1])}
expected = json.loads("{\"regression_score_name\":\"R2\",\"regression_score\":0.5555555555555556,\"classification_score_name\":\"accuracy\",\"classification_score\":0.5}")
assert equivalent(normalize(result),expected)
results["S02.O3"] = normalize(result)
print(json.dumps({"objective":"S02.O3","output":normalize(result)},sort_keys=True))

### S02.O4: Separate improved validation metrics from causal/calibrated guarantees.
Predict first: Select a limited conclusion from changed prediction vectors.

A validation error describes agreement on specified observations. It does not identify an intervention effect, prove calibrated probabilities or guarantee performance for a different population.

Misconception: A lower validation error proves a causal effect, calibrated probabilities and superiority in every population.

Remediation: List which claims the data and metric actually measure.

Guided change: Compute regression units, then inspect a majority-class baseline's missed positives. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
truth = np.array([100.,200.]); prediction = np.array([120.,180.])
result = {'validation_mae':mean_absolute_error(truth,prediction),'causal_effect_identified':False,'calibrated_probabilities_proved':False,'claim':'describes these declared prediction errors only'}
expected = json.loads("{\"validation_mae\":20,\"causal_effect_identified\":false,\"calibrated_probabilities_proved\":false,\"claim\":\"describes these declared prediction errors only\"}")
assert equivalent(normalize(result),expected)
results["S02.O4"] = normalize(result)
print(json.dumps({"objective":"S02.O4","output":normalize(result)},sort_keys=True))

## S03 — Cross-validation keeps validation inside training
Prerequisites/recall: S01–S02, T05.

Cross-validation repeats train/validate boundaries inside the training set, not on the final test.

Repeated validation summaries; overlapping folds are not independent population experiments.

Terminology: fold, KFold, GroupKFold, TimeSeriesSplit, cross_val_score, negative scorer.

A fold is a temporary training/validation partition inside development data. Cross-validation repeats that local experiment to compare a stated workflow; it never turns the final test into training data. Each learned imputer, scaler, vocabulary and model must be fitted anew using only that fold's training IDs. Reuse the ID arrows from T05, not a globally prefit transformer.

scikit-learn's negative loss scorers follow a larger-is-better convention. neg_mean_absolute_error values−10,−20,−30 mean actual MAEs10,20,30 and average20 in target units. A more negative score is worse for this scorer. The mean is a summary of the declared folds, not a negative physical error or an independent population experiment.

Choose the unit from the prediction-time question. KFold with shuffle and seed42 can suit independent rows; GroupKFold keeps customer groups separate; TimeSeriesSplit puts training before later validation observations. A seed does not fix the wrong split unit or ensure representative data. Small folds and overlapping training sets limit confidence; no formal interval, universal ranking or causal effect is inferred from three numbers.

Progressive experiment: Inspect one fold, then all folds, scorer sign and a group/time counterexample.

### S03.O1 — critical decision AND reason: Keep validation/final-test rows out of fold fitting with the correct split unit.
Predict first: Reject a leaking or group/time-inappropriate fold plan and explain the boundary.

Each fold fits every learned preprocessing/model stage on its training IDs only. Validation and final-test rows cannot learn state. Random folds suit independent rows; groups require entity separation and time forecasts require earlier training observations.

Misconception: Cross-validation permits fitting preprocessing on all development rows or including final-test rows, regardless of group or time.

Remediation: Color-independent fit/validate/test ID tables.

Guided change: Inspect one fold, then all folds, scorer sign and a group/time counterexample. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
ids = np.array(['H01','H02','H03','H04','H05','H06']); groups = np.array(['A','A','B','B','C','C'])
features = np.array([[1.],[np.nan],[3.],[4.],[5.],[9.]]); targets = np.array([10.,20.,30.,40.,50.,60.])
splitter = KFold(n_splits=3,shuffle=True,random_state=42); ledger = []
for fitting, validating in splitter.split(features,targets,groups if False else None):
    pipeline = make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),DummyRegressor(strategy='mean')).fit(features[fitting],targets[fitting])
    ledger.append({'fit_ids':ids[fitting].tolist(),'validation_ids':ids[validating].tolist(),'fit_groups':groups[fitting].tolist(),'validation_groups':groups[validating].tolist(),'training_median':pipeline.named_steps['simpleimputer'].statistics_,'validation_predictions':pipeline.predict(features[validating])})
result = {'scenario':'random','splitter':type(splitter).__name__,'folds':ledger,'final_test_ids':['H13','H14','H15','H16'],'prefit_on_all_rows_allowed':False}

expected = json.loads("{\"scenario\":\"random\",\"splitter\":\"KFold\",\"folds\":[{\"fit_ids\":[\"H03\",\"H04\",\"H05\",\"H06\"],\"validation_ids\":[\"H01\",\"H02\"],\"fit_groups\":[\"B\",\"B\",\"C\",\"C\"],\"validation_groups\":[\"A\",\"A\"],\"training_median\":[4.5],\"validation_predictions\":[45,45]},{\"fit_ids\":[\"H01\",\"H02\",\"H04\",\"H05\"],\"validation_ids\":[\"H03\",\"H06\"],\"fit_groups\":[\"A\",\"A\",\"B\",\"C\"],\"validation_groups\":[\"B\",\"C\"],\"training_median\":[4],\"validation_predictions\":[30,30]},{\"fit_ids\":[\"H01\",\"H02\",\"H03\",\"H06\"],\"validation_ids\":[\"H04\",\"H05\"],\"fit_groups\":[\"A\",\"A\",\"B\",\"C\"],\"validation_groups\":[\"B\",\"C\"],\"training_median\":[3],\"validation_predictions\":[30,30]}],\"final_test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"prefit_on_all_rows_allowed\":false}")
assert equivalent(normalize(result),expected)
results["S03.O1"] = normalize(result)
print(json.dumps({"objective":"S03.O1","output":normalize(result)},sort_keys=True))

### S03.O2: Interpret negative loss scorers and aggregate declared fold scores.
Predict first: Select mean MAE from signed scorer outputs.

The negative-loss scorer is larger-is-better for API consistency. Negate it before reporting MAE in target units; mean of positive MAEs is not a negative physical error.

Misconception: A negative MAE scorer is a physically negative error, so a more negative score always means a better model.

Remediation: Convert the sign before reporting units.

Guided change: Inspect one fold, then all folds, scorer sign and a group/time counterexample. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
signed_scores = np.array([-10.,-20.,-30.]) + 0.
result = {'negative_mae_scores':signed_scores,'fold_mae':-signed_scores,'mean_mae':float((-signed_scores).mean()),'units':'$k'}
expected = json.loads("{\"negative_mae_scores\":[-10,-20,-30],\"fold_mae\":[10,20,30],\"mean_mae\":20,\"units\":\"$k\"}")
assert equivalent(normalize(result),expected)
results["S03.O2"] = normalize(result)
print(json.dumps({"objective":"S03.O2","output":normalize(result)},sort_keys=True))

### S03.O3: Recognize limitations of small-fold variability.
Predict first: Reject a universal-confidence claim from a few fold scores.

A few fold errors summarize these small overlapping partitions, not independent population experiments. Variation flags uncertainty and data assumptions; it does not provide a universal guarantee or formal confidence interval.

Misconception: Three folds are independent population experiments that certify universal confidence and model superiority.

Remediation: Compare sample sizes, overlap and data assumptions.

Guided change: Inspect one fold, then all folds, scorer sign and a group/time counterexample. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
features = np.arange(6).reshape(-1,1); target = np.array([1.,2.,3.,4.,103.,200.])
scores = -cross_val_score(DummyRegressor(strategy='mean'),features,target,cv=KFold(n_splits=3,shuffle=True,random_state=42),scoring='neg_mean_absolute_error',n_jobs=1)
result = {'fold_mae':scores,'mean_mae':float(scores.mean()),'overlapping_training_sets':True,'universal_confidence_guaranteed':False}
expected = json.loads("{\"fold_mae\":[76,98.5,49.5],\"mean_mae\":74.66666666666667,\"overlapping_training_sets\":true,\"universal_confidence_guaranteed\":false}")
assert equivalent(normalize(result),expected)
results["S03.O3"] = normalize(result)
print(json.dumps({"objective":"S03.O3","output":normalize(result)},sort_keys=True))

### S03.O4: Choose reproducible splitter settings without assuming shuffling always helps.
Predict first: Match time/group/random scenarios to configured splitters.

Choose the splitter from the intended generalization question. Fixed seed42 reproduces this random split for the same data/order; GroupKFold separates customer groups and TimeSeriesSplit preserves chronology without inappropriate shuffling/random_state.

Misconception: Shuffling always improves evaluation, and setting a seed makes group/time overlap irrelevant.

Remediation: Review the future-use question and entity/time order.

Guided change: Inspect one fold, then all folds, scorer sign and a group/time counterexample. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
ids = np.array(['H01','H02','H03','H04','H05','H06']); groups = np.array(['A','A','B','B','C','C'])
features = np.array([[1.],[np.nan],[3.],[4.],[5.],[9.]]); targets = np.array([10.,20.,30.,40.,50.,60.])
splitter = KFold(n_splits=3,shuffle=True,random_state=42); ledger = []
for fitting, validating in splitter.split(features,targets,groups if False else None):
    pipeline = make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),DummyRegressor(strategy='mean')).fit(features[fitting],targets[fitting])
    ledger.append({'fit_ids':ids[fitting].tolist(),'validation_ids':ids[validating].tolist(),'fit_groups':groups[fitting].tolist(),'validation_groups':groups[validating].tolist(),'training_median':pipeline.named_steps['simpleimputer'].statistics_,'validation_predictions':pipeline.predict(features[validating])})
result = {'scenario':'random','splitter':type(splitter).__name__,'folds':ledger,'final_test_ids':['H13','H14','H15','H16'],'prefit_on_all_rows_allowed':False}

expected = json.loads("{\"scenario\":\"random\",\"splitter\":\"KFold\",\"folds\":[{\"fit_ids\":[\"H03\",\"H04\",\"H05\",\"H06\"],\"validation_ids\":[\"H01\",\"H02\"],\"fit_groups\":[\"B\",\"B\",\"C\",\"C\"],\"validation_groups\":[\"A\",\"A\"],\"training_median\":[4.5],\"validation_predictions\":[45,45]},{\"fit_ids\":[\"H01\",\"H02\",\"H04\",\"H05\"],\"validation_ids\":[\"H03\",\"H06\"],\"fit_groups\":[\"A\",\"A\",\"B\",\"C\"],\"validation_groups\":[\"B\",\"C\"],\"training_median\":[4],\"validation_predictions\":[30,30]},{\"fit_ids\":[\"H01\",\"H02\",\"H03\",\"H06\"],\"validation_ids\":[\"H04\",\"H05\"],\"fit_groups\":[\"A\",\"A\",\"B\",\"C\"],\"validation_groups\":[\"B\",\"C\"],\"training_median\":[3],\"validation_predictions\":[30,30]}],\"final_test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"prefit_on_all_rows_allowed\":false}")
assert equivalent(normalize(result),expected)
results["S03.O4"] = normalize(result)
print(json.dumps({"objective":"S03.O4","output":normalize(result)},sort_keys=True))

## S04 — A complete train-fitted pipeline
Prerequisites/recall: S01–S03, T05–T06; existing Linear Regression.

A complete pipeline is composed familiar contracts, not a new block of magic code.

Compose fitted coordinate transforms and the existing linear prediction model.

Terminology: make_pipeline/Pipeline, ColumnTransformer, SimpleImputer, OneHotEncoder, StandardScaler, LinearRegression.

A complete pipeline composes familiar contracts: pandas names and row identity, training-fitted missingness, NumPy coordinate arrays and the existing linear model. First inspect the numeric branch: median imputation then population scaling. Next the categorical branch: declared label imputation then fixed A/B one-hot coordinates. ColumnTransformer concatenates branches in declaration order; feature names matter as much as shape.

Cross-validation must receive this whole estimator, not only a final model after preprocessing all development data. For each fold, read the fit IDs and inspect its learned median, scaling mean and category vocabulary. A globally prefit pipeline leaks validation information even when its numerical prediction happens to match a valid fit. A future-derived feature is already leaked before the pipeline sees it.

After the workflow is selected using development-only validation, fit the selected full pipeline on the12 permitted development rows and predict the four untouched final-test rows in their exact ID order. The preserved Linear Regression connection is ŷ=Xw+b after the stated transformations; this does not change the existing Regression runtime. A trusted notebook records actual fitted predictions and environment; the browser's frozen simulation does not train your model or certify population performance.

Progressive experiment: Trace numeric/categorical branches separately before the complete fold-local model.

### Two branch traces before the complete estimator
Work each branch numerically before the complete pipeline. Numeric training[10,missing,30] becomes[10,20,30] under its training median20. Its mean is20, population variance200/3 and scale sqrt(200/3), so the scaled column is approximately[-1.224744871,0,1.224744871]. Categorical training[A,B,missing] becomes[A,B,A] under its declared most-frequent rule, then fixed columns[A,B] give[[1,0],[0,1],[1,0]]. Concatenate numeric first, then the two named category columns; the model receives three coordinates per row, not newly learned test categories.

In [ ]:
numeric_branch = make_pipeline(SimpleImputer(strategy="median"),StandardScaler()).fit_transform([[10.],[np.nan],[30.]])
category_branch = make_pipeline(SimpleImputer(strategy="most_frequent"),OneHotEncoder(categories=[["A","B"]],handle_unknown="ignore",sparse_output=False)).fit_transform(np.array([["A"],["B"],[np.nan]],dtype=object))
np.testing.assert_allclose(numeric_branch[:,0],[-np.sqrt(3/2),0,np.sqrt(3/2)])
np.testing.assert_array_equal(category_branch,[[1,0],[0,1],[1,0]])
print(json.dumps({"numeric":normalize(numeric_branch),"category":normalize(category_branch),"combined":normalize(np.hstack([numeric_branch,category_branch]))},sort_keys=True))

### S04.O1 — critical decision AND reason: Place all learned preprocessing inside the cross-validated estimator.
Predict first: Repair external fitting and explain fold-local state.

Cross-validation must receive the entire train-fitted preprocessing plus model. Every stage learns only each fold's training rows. Fitting a preprocessor globally first leaks validation information even if the model itself is cross-validated.

Misconception: Preprocessing is unsupervised, so it may be fitted globally before cross-validating the final model.

Remediation: Revisit T05 arrows and S03 fold memberships.

Guided change: Trace numeric/categorical branches separately before the complete fold-local model. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
houses = pd.DataFrame({'id': ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16'], 'area': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], 'age': [0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10], 'neighborhood': ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B'], 'price': [100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400]})
features = ['area','age','neighborhood']; train = houses.iloc[:12]; test = houses.iloc[12:].iloc[[0, 1, 2, 3]]
def candidate(model):
    return make_pipeline(ColumnTransformer([('numeric',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),['area','age']),('category',make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(categories=[['A','B']],handle_unknown='ignore',sparse_output=False)),['neighborhood'])]),model)
folds = list(KFold(n_splits=3,shuffle=True,random_state=42).split(train)); scores = {}
for name, model in [('dummy',DummyRegressor(strategy='mean')),('linear',LinearRegression())]:
    scores[name] = -cross_val_score(candidate(model),train[features],train['price'],cv=folds,scoring='neg_mean_absolute_error',n_jobs=1)
ledger = []
for fitting, validating in folds:
    fitted = candidate(LinearRegression()).fit(train.iloc[fitting][features],train.iloc[fitting]['price']); preparation = fitted.steps[0][1]
    numeric = preparation.named_transformers_['numeric']; category = preparation.named_transformers_['category']
    ledger.append({'fit_ids':train.iloc[fitting]['id'].tolist(),'validation_ids':train.iloc[validating]['id'].tolist(),'imputer':numeric.named_steps['simpleimputer'].statistics_,'scaler_mean':numeric.named_steps['standardscaler'].mean_,'categories':category.named_steps['onehotencoder'].categories_})
means = {name:float(values.mean()) for name,values in scores.items()}; tied = abs(means['linear']-means['dummy']) <= 1e-9+1e-9*abs(means['dummy']); winner = 'dummy' if tied or means['dummy'] < means['linear'] else 'linear'
chosen = candidate(DummyRegressor(strategy='mean') if winner == 'dummy' else LinearRegression()).fit(train[features],train['price']); prediction = chosen.predict(test[features])
result = {'dataset':'sklearn-houses.v1','data_sha256':'751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5','environment':'macos-arm64-cpython-3.13.16-reference.1','seed':42,'candidate_parameters':['DummyRegressor(mean)','LinearRegression(default)'],'folds':ledger,'cv_mae':scores,'mean_cv_mae':means,'tied':tied,'selected_before_test':winner,'refit_ids':train['id'].tolist(),'test_ids':test['id'].tolist(),'predictions':prediction,'columns':chosen.steps[0][1].get_feature_names_out(),'mae':mean_absolute_error(test['price'],prediction),'mse':mean_squared_error(test['price'],prediction),'rmse':root_mean_squared_error(test['price'],prediction),'test_may_select':False,'browser_trained':False}
result['question'] = 'fold-local learned state'

expected = json.loads("{\"dataset\":\"sklearn-houses.v1\",\"data_sha256\":\"751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5\",\"environment\":\"macos-arm64-cpython-3.13.16-reference.1\",\"seed\":42,\"candidate_parameters\":[\"DummyRegressor(mean)\",\"LinearRegression(default)\"],\"folds\":[{\"fit_ids\":[\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H12\"],\"validation_ids\":[\"H01\",\"H09\",\"H10\",\"H11\"],\"imputer\":[5.5,7.5],\"scaler_mean\":[5.875,6.875],\"categories\":[[\"A\",\"B\"]]},{\"fit_ids\":[\"H01\",\"H04\",\"H05\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\"],\"validation_ids\":[\"H02\",\"H03\",\"H06\",\"H12\"],\"imputer\":[7.5,2.5],\"scaler_mean\":[6.875,3.75],\"categories\":[[\"A\",\"B\"]]},{\"fit_ids\":[\"H01\",\"H02\",\"H03\",\"H06\",\"H09\",\"H10\",\"H11\",\"H12\"],\"validation_ids\":[\"H04\",\"H05\",\"H07\",\"H08\"],\"imputer\":[7.5,5],\"scaler_mean\":[6.75,4.375],\"categories\":[[\"A\",\"B\"]]}],\"cv_mae\":{\"dummy\":[86.25,73.75,30],\"linear\":[1.0658141036401503e-14,3.552713678800501e-15,2.842170943040401e-14]},\"mean_cv_mae\":{\"dummy\":63.333333333333336,\"linear\":1.4210854715202004e-14},\"tied\":false,\"selected_before_test\":\"linear\",\"refit_ids\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\"],\"test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"predictions\":[340,360.00000000000006,380,400.00000000000006],\"columns\":[\"numeric__area\",\"numeric__age\",\"category__neighborhood_A\",\"category__neighborhood_B\"],\"mae\":2.842170943040401e-14,\"mse\":1.6155871338926322e-27,\"rmse\":4.0194366942304644e-14,\"test_may_select\":false,\"browser_trained\":false,\"question\":\"fold-local learned state\"}")
assert equivalent(normalize(result),expected)
results["S04.O1"] = normalize(result)
print(json.dumps({"objective":"S04.O1","output":normalize(result)},sort_keys=True))

### S04.O2: Select valid ColumnTransformer branches and explicit encoder configuration.
Predict first: Choose a complete allowlisted pipeline repair.

Numeric age/area use training-fitted imputation then scaling; the categorical branch imputes labels then encodes the saved A/B vocabulary. Declared dense output, unknown policy and named column order define the input contract.

Misconception: Any branch order or encoder vocabulary is valid if the matrix has the same width.

Remediation: Trace numeric/categorical types and output names.

Guided change: Trace numeric/categorical branches separately before the complete fold-local model. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
houses = pd.DataFrame({'id': ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16'], 'area': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], 'age': [0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10], 'neighborhood': ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B'], 'price': [100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400]})
features = ['area','age','neighborhood']; train = houses.iloc[:12]; test = houses.iloc[12:].iloc[[0, 1, 2, 3]]
def candidate(model):
    return make_pipeline(ColumnTransformer([('numeric',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),['area','age']),('category',make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(categories=[['A','B']],handle_unknown='ignore',sparse_output=False)),['neighborhood'])]),model)
folds = list(KFold(n_splits=3,shuffle=True,random_state=42).split(train)); scores = {}
for name, model in [('dummy',DummyRegressor(strategy='mean')),('linear',LinearRegression())]:
    scores[name] = -cross_val_score(candidate(model),train[features],train['price'],cv=folds,scoring='neg_mean_absolute_error',n_jobs=1)
ledger = []
for fitting, validating in folds:
    fitted = candidate(LinearRegression()).fit(train.iloc[fitting][features],train.iloc[fitting]['price']); preparation = fitted.steps[0][1]
    numeric = preparation.named_transformers_['numeric']; category = preparation.named_transformers_['category']
    ledger.append({'fit_ids':train.iloc[fitting]['id'].tolist(),'validation_ids':train.iloc[validating]['id'].tolist(),'imputer':numeric.named_steps['simpleimputer'].statistics_,'scaler_mean':numeric.named_steps['standardscaler'].mean_,'categories':category.named_steps['onehotencoder'].categories_})
means = {name:float(values.mean()) for name,values in scores.items()}; tied = abs(means['linear']-means['dummy']) <= 1e-9+1e-9*abs(means['dummy']); winner = 'dummy' if tied or means['dummy'] < means['linear'] else 'linear'
chosen = candidate(DummyRegressor(strategy='mean') if winner == 'dummy' else LinearRegression()).fit(train[features],train['price']); prediction = chosen.predict(test[features])
result = {'dataset':'sklearn-houses.v1','data_sha256':'751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5','environment':'macos-arm64-cpython-3.13.16-reference.1','seed':42,'candidate_parameters':['DummyRegressor(mean)','LinearRegression(default)'],'folds':ledger,'cv_mae':scores,'mean_cv_mae':means,'tied':tied,'selected_before_test':winner,'refit_ids':train['id'].tolist(),'test_ids':test['id'].tolist(),'predictions':prediction,'columns':chosen.steps[0][1].get_feature_names_out(),'mae':mean_absolute_error(test['price'],prediction),'mse':mean_squared_error(test['price'],prediction),'rmse':root_mean_squared_error(test['price'],prediction),'test_may_select':False,'browser_trained':False}
result['question'] = 'typed branch and vocabulary contract'

expected = json.loads("{\"dataset\":\"sklearn-houses.v1\",\"data_sha256\":\"751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5\",\"environment\":\"macos-arm64-cpython-3.13.16-reference.1\",\"seed\":42,\"candidate_parameters\":[\"DummyRegressor(mean)\",\"LinearRegression(default)\"],\"folds\":[{\"fit_ids\":[\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H12\"],\"validation_ids\":[\"H01\",\"H09\",\"H10\",\"H11\"],\"imputer\":[5.5,7.5],\"scaler_mean\":[5.875,6.875],\"categories\":[[\"A\",\"B\"]]},{\"fit_ids\":[\"H01\",\"H04\",\"H05\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\"],\"validation_ids\":[\"H02\",\"H03\",\"H06\",\"H12\"],\"imputer\":[7.5,2.5],\"scaler_mean\":[6.875,3.75],\"categories\":[[\"A\",\"B\"]]},{\"fit_ids\":[\"H01\",\"H02\",\"H03\",\"H06\",\"H09\",\"H10\",\"H11\",\"H12\"],\"validation_ids\":[\"H04\",\"H05\",\"H07\",\"H08\"],\"imputer\":[7.5,5],\"scaler_mean\":[6.75,4.375],\"categories\":[[\"A\",\"B\"]]}],\"cv_mae\":{\"dummy\":[86.25,73.75,30],\"linear\":[1.0658141036401503e-14,3.552713678800501e-15,2.842170943040401e-14]},\"mean_cv_mae\":{\"dummy\":63.333333333333336,\"linear\":1.4210854715202004e-14},\"tied\":false,\"selected_before_test\":\"linear\",\"refit_ids\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\"],\"test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"predictions\":[340,360.00000000000006,380,400.00000000000006],\"columns\":[\"numeric__area\",\"numeric__age\",\"category__neighborhood_A\",\"category__neighborhood_B\"],\"mae\":2.842170943040401e-14,\"mse\":1.6155871338926322e-27,\"rmse\":4.0194366942304644e-14,\"test_may_select\":false,\"browser_trained\":false,\"question\":\"typed branch and vocabulary contract\"}")
assert equivalent(normalize(result),expected)
results["S04.O2"] = normalize(result)
print(json.dumps({"objective":"S04.O2","output":normalize(result)},sort_keys=True))

### S04.O3: Predict the fit/predict output contract and compare trusted notebook results.
Predict first: Choose expected result length/schema and fixture agreement.

Fit learns state from permitted training rows; predict returns one numeric output per supplied held-out row in the same ID order. A frozen trace must agree with actual notebook values within declared tolerance and preserve feature names.

Misconception: Predictions may be reordered or refitted on evaluation rows as long as the output length is correct.

Remediation: Map held-out row IDs to prediction positions.

Guided change: Trace numeric/categorical branches separately before the complete fold-local model. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
houses = pd.DataFrame({'id': ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16'], 'area': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], 'age': [0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10], 'neighborhood': ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B'], 'price': [100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400]})
features = ['area','age','neighborhood']; train = houses.iloc[:12]; test = houses.iloc[12:].iloc[[0, 1, 2, 3]]
def candidate(model):
    return make_pipeline(ColumnTransformer([('numeric',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),['area','age']),('category',make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(categories=[['A','B']],handle_unknown='ignore',sparse_output=False)),['neighborhood'])]),model)
folds = list(KFold(n_splits=3,shuffle=True,random_state=42).split(train)); scores = {}
for name, model in [('dummy',DummyRegressor(strategy='mean')),('linear',LinearRegression())]:
    scores[name] = -cross_val_score(candidate(model),train[features],train['price'],cv=folds,scoring='neg_mean_absolute_error',n_jobs=1)
ledger = []
for fitting, validating in folds:
    fitted = candidate(LinearRegression()).fit(train.iloc[fitting][features],train.iloc[fitting]['price']); preparation = fitted.steps[0][1]
    numeric = preparation.named_transformers_['numeric']; category = preparation.named_transformers_['category']
    ledger.append({'fit_ids':train.iloc[fitting]['id'].tolist(),'validation_ids':train.iloc[validating]['id'].tolist(),'imputer':numeric.named_steps['simpleimputer'].statistics_,'scaler_mean':numeric.named_steps['standardscaler'].mean_,'categories':category.named_steps['onehotencoder'].categories_})
means = {name:float(values.mean()) for name,values in scores.items()}; tied = abs(means['linear']-means['dummy']) <= 1e-9+1e-9*abs(means['dummy']); winner = 'dummy' if tied or means['dummy'] < means['linear'] else 'linear'
chosen = candidate(DummyRegressor(strategy='mean') if winner == 'dummy' else LinearRegression()).fit(train[features],train['price']); prediction = chosen.predict(test[features])
result = {'dataset':'sklearn-houses.v1','data_sha256':'751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5','environment':'macos-arm64-cpython-3.13.16-reference.1','seed':42,'candidate_parameters':['DummyRegressor(mean)','LinearRegression(default)'],'folds':ledger,'cv_mae':scores,'mean_cv_mae':means,'tied':tied,'selected_before_test':winner,'refit_ids':train['id'].tolist(),'test_ids':test['id'].tolist(),'predictions':prediction,'columns':chosen.steps[0][1].get_feature_names_out(),'mae':mean_absolute_error(test['price'],prediction),'mse':mean_squared_error(test['price'],prediction),'rmse':root_mean_squared_error(test['price'],prediction),'test_may_select':False,'browser_trained':False}
result['question'] = 'ID-aligned fitted predictions'

expected = json.loads("{\"dataset\":\"sklearn-houses.v1\",\"data_sha256\":\"751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5\",\"environment\":\"macos-arm64-cpython-3.13.16-reference.1\",\"seed\":42,\"candidate_parameters\":[\"DummyRegressor(mean)\",\"LinearRegression(default)\"],\"folds\":[{\"fit_ids\":[\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H12\"],\"validation_ids\":[\"H01\",\"H09\",\"H10\",\"H11\"],\"imputer\":[5.5,7.5],\"scaler_mean\":[5.875,6.875],\"categories\":[[\"A\",\"B\"]]},{\"fit_ids\":[\"H01\",\"H04\",\"H05\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\"],\"validation_ids\":[\"H02\",\"H03\",\"H06\",\"H12\"],\"imputer\":[7.5,2.5],\"scaler_mean\":[6.875,3.75],\"categories\":[[\"A\",\"B\"]]},{\"fit_ids\":[\"H01\",\"H02\",\"H03\",\"H06\",\"H09\",\"H10\",\"H11\",\"H12\"],\"validation_ids\":[\"H04\",\"H05\",\"H07\",\"H08\"],\"imputer\":[7.5,5],\"scaler_mean\":[6.75,4.375],\"categories\":[[\"A\",\"B\"]]}],\"cv_mae\":{\"dummy\":[86.25,73.75,30],\"linear\":[1.0658141036401503e-14,3.552713678800501e-15,2.842170943040401e-14]},\"mean_cv_mae\":{\"dummy\":63.333333333333336,\"linear\":1.4210854715202004e-14},\"tied\":false,\"selected_before_test\":\"linear\",\"refit_ids\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\"],\"test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"predictions\":[340,360.00000000000006,380,400.00000000000006],\"columns\":[\"numeric__area\",\"numeric__age\",\"category__neighborhood_A\",\"category__neighborhood_B\"],\"mae\":2.842170943040401e-14,\"mse\":1.6155871338926322e-27,\"rmse\":4.0194366942304644e-14,\"test_may_select\":false,\"browser_trained\":false,\"question\":\"ID-aligned fitted predictions\"}")
assert equivalent(normalize(result),expected)
results["S04.O3"] = normalize(result)
print(json.dumps({"objective":"S04.O3","output":normalize(result)},sort_keys=True))

### S04.O4: Report execution, metrics and limitations honestly.
Predict first: Reject browser-simulation-as-training or synthetic-generalization claims.

This trusted author reference actually fits the model; the browser only displays its frozen trace. Report exact data, splits, metric and execution manifest, and the limitations of fictional teaching observations rather than claiming population performance.

Misconception: Displaying synthetic predictions in the browser proves live training and real-world generalization.

Remediation: Compare the generated fixture and actual notebook run record.

Guided change: Trace numeric/categorical branches separately before the complete fold-local model. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
houses = pd.DataFrame({'id': ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16'], 'area': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], 'age': [0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10], 'neighborhood': ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B'], 'price': [100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400]})
features = ['area','age','neighborhood']; train = houses.iloc[:12]; test = houses.iloc[12:].iloc[[0, 1, 2, 3]]
def candidate(model):
    return make_pipeline(ColumnTransformer([('numeric',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),['area','age']),('category',make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(categories=[['A','B']],handle_unknown='ignore',sparse_output=False)),['neighborhood'])]),model)
folds = list(KFold(n_splits=3,shuffle=True,random_state=42).split(train)); scores = {}
for name, model in [('dummy',DummyRegressor(strategy='mean')),('linear',LinearRegression())]:
    scores[name] = -cross_val_score(candidate(model),train[features],train['price'],cv=folds,scoring='neg_mean_absolute_error',n_jobs=1)
ledger = []
for fitting, validating in folds:
    fitted = candidate(LinearRegression()).fit(train.iloc[fitting][features],train.iloc[fitting]['price']); preparation = fitted.steps[0][1]
    numeric = preparation.named_transformers_['numeric']; category = preparation.named_transformers_['category']
    ledger.append({'fit_ids':train.iloc[fitting]['id'].tolist(),'validation_ids':train.iloc[validating]['id'].tolist(),'imputer':numeric.named_steps['simpleimputer'].statistics_,'scaler_mean':numeric.named_steps['standardscaler'].mean_,'categories':category.named_steps['onehotencoder'].categories_})
means = {name:float(values.mean()) for name,values in scores.items()}; tied = abs(means['linear']-means['dummy']) <= 1e-9+1e-9*abs(means['dummy']); winner = 'dummy' if tied or means['dummy'] < means['linear'] else 'linear'
chosen = candidate(DummyRegressor(strategy='mean') if winner == 'dummy' else LinearRegression()).fit(train[features],train['price']); prediction = chosen.predict(test[features])
result = {'dataset':'sklearn-houses.v1','data_sha256':'751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5','environment':'macos-arm64-cpython-3.13.16-reference.1','seed':42,'candidate_parameters':['DummyRegressor(mean)','LinearRegression(default)'],'folds':ledger,'cv_mae':scores,'mean_cv_mae':means,'tied':tied,'selected_before_test':winner,'refit_ids':train['id'].tolist(),'test_ids':test['id'].tolist(),'predictions':prediction,'columns':chosen.steps[0][1].get_feature_names_out(),'mae':mean_absolute_error(test['price'],prediction),'mse':mean_squared_error(test['price'],prediction),'rmse':root_mean_squared_error(test['price'],prediction),'test_may_select':False,'browser_trained':False}
result['question'] = 'execution and limitations'

expected = json.loads("{\"dataset\":\"sklearn-houses.v1\",\"data_sha256\":\"751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5\",\"environment\":\"macos-arm64-cpython-3.13.16-reference.1\",\"seed\":42,\"candidate_parameters\":[\"DummyRegressor(mean)\",\"LinearRegression(default)\"],\"folds\":[{\"fit_ids\":[\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H12\"],\"validation_ids\":[\"H01\",\"H09\",\"H10\",\"H11\"],\"imputer\":[5.5,7.5],\"scaler_mean\":[5.875,6.875],\"categories\":[[\"A\",\"B\"]]},{\"fit_ids\":[\"H01\",\"H04\",\"H05\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\"],\"validation_ids\":[\"H02\",\"H03\",\"H06\",\"H12\"],\"imputer\":[7.5,2.5],\"scaler_mean\":[6.875,3.75],\"categories\":[[\"A\",\"B\"]]},{\"fit_ids\":[\"H01\",\"H02\",\"H03\",\"H06\",\"H09\",\"H10\",\"H11\",\"H12\"],\"validation_ids\":[\"H04\",\"H05\",\"H07\",\"H08\"],\"imputer\":[7.5,5],\"scaler_mean\":[6.75,4.375],\"categories\":[[\"A\",\"B\"]]}],\"cv_mae\":{\"dummy\":[86.25,73.75,30],\"linear\":[1.0658141036401503e-14,3.552713678800501e-15,2.842170943040401e-14]},\"mean_cv_mae\":{\"dummy\":63.333333333333336,\"linear\":1.4210854715202004e-14},\"tied\":false,\"selected_before_test\":\"linear\",\"refit_ids\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\"],\"test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"predictions\":[340,360.00000000000006,380,400.00000000000006],\"columns\":[\"numeric__area\",\"numeric__age\",\"category__neighborhood_A\",\"category__neighborhood_B\"],\"mae\":2.842170943040401e-14,\"mse\":1.6155871338926322e-27,\"rmse\":4.0194366942304644e-14,\"test_may_select\":false,\"browser_trained\":false,\"question\":\"execution and limitations\"}")
assert equivalent(normalize(result),expected)
results["S04.O4"] = normalize(result)
print(json.dumps({"objective":"S04.O4","output":normalize(result)},sort_keys=True))

## S05 — Compare models without choosing on the test set
Prerequisites/recall: S02–S04; Optimization validation.

Selection and final evaluation are different experiments and must remain separated.

Selection bias; one held-out score does not establish universal superiority.

Terminology: model comparison, selection, validation protocol, refit, final evaluation, uncertainty.

Selection and final evaluation answer different questions. Fix the candidates, folds and metric before comparison: the simple mean baseline versus the existing linear-regression pipeline, judged by training-only mean CV MAE. The final test estimates the already selected workflow; using it to choose a winner makes it part of development.

Compare identical folds and positive MAEs. Lower is better; if means are within the declared absolute1e−9 plus relative1e−9 tolerance, prefer the simpler DummyRegressor as the approved tie rule. This is a predefined decision convention, not proof that tied models have identical real-world utility. Refit only the selected frozen workflow on all allowed development rows, then evaluate once on the final test without changing the choice.

A reproducible report includes data/hash, feature order, versions, seeds, candidate parameters, signed and normalized fold scores, per-stage fit IDs, tie/selection rule, refit/test membership, predictions, metrics and limitations. A tiny fictional CV win is not universal superiority or deployment authorization. S05.O4 introduces this uncertainty cautiously, not a new inference algorithm. Connect the report to Statistics sampling, Visualization honest claims and Data Preparation provenance; broader evaluation and capstones require later approval.

Progressive experiment: Apply a fixed validation rule, then refit and inspect a declared tie/limited claim.

### S05.O1 — critical decision AND reason: Select candidates without using final-test feedback.
Predict first: Repair a test-guided winner selection and justify validation-only choice.

Choose the candidate using training-only CV MAE on identical folds, freeze the decision, refit on development training and only then evaluate the final test. Test feedback must not select either the model or preprocessing policy.

Misconception: Evaluate all candidates on the final test and choose its best score, then report the same test as unbiased.

Remediation: Draw the selection/refit/final-test timeline.

Guided change: Apply a fixed validation rule, then refit and inspect a declared tie/limited claim. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
houses = pd.DataFrame({'id': ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16'], 'area': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], 'age': [0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10], 'neighborhood': ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B'], 'price': [100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400]})
features = ['area','age','neighborhood']; train = houses.iloc[:12]; test = houses.iloc[12:].iloc[[0, 1, 2, 3]]
def candidate(model):
    return make_pipeline(ColumnTransformer([('numeric',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),['area','age']),('category',make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(categories=[['A','B']],handle_unknown='ignore',sparse_output=False)),['neighborhood'])]),model)
folds = list(KFold(n_splits=3,shuffle=True,random_state=42).split(train)); scores = {}
for name, model in [('dummy',DummyRegressor(strategy='mean')),('linear',LinearRegression())]:
    scores[name] = -cross_val_score(candidate(model),train[features],train['price'],cv=folds,scoring='neg_mean_absolute_error',n_jobs=1)
ledger = []
for fitting, validating in folds:
    fitted = candidate(LinearRegression()).fit(train.iloc[fitting][features],train.iloc[fitting]['price']); preparation = fitted.steps[0][1]
    numeric = preparation.named_transformers_['numeric']; category = preparation.named_transformers_['category']
    ledger.append({'fit_ids':train.iloc[fitting]['id'].tolist(),'validation_ids':train.iloc[validating]['id'].tolist(),'imputer':numeric.named_steps['simpleimputer'].statistics_,'scaler_mean':numeric.named_steps['standardscaler'].mean_,'categories':category.named_steps['onehotencoder'].categories_})
means = {name:float(values.mean()) for name,values in scores.items()}; tied = abs(means['linear']-means['dummy']) <= 1e-9+1e-9*abs(means['dummy']); winner = 'dummy' if tied or means['dummy'] < means['linear'] else 'linear'
chosen = candidate(DummyRegressor(strategy='mean') if winner == 'dummy' else LinearRegression()).fit(train[features],train['price']); prediction = chosen.predict(test[features])
result = {'dataset':'sklearn-houses.v1','data_sha256':'751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5','environment':'macos-arm64-cpython-3.13.16-reference.1','seed':42,'candidate_parameters':['DummyRegressor(mean)','LinearRegression(default)'],'folds':ledger,'cv_mae':scores,'mean_cv_mae':means,'tied':tied,'selected_before_test':winner,'refit_ids':train['id'].tolist(),'test_ids':test['id'].tolist(),'predictions':prediction,'columns':chosen.steps[0][1].get_feature_names_out(),'mae':mean_absolute_error(test['price'],prediction),'mse':mean_squared_error(test['price'],prediction),'rmse':root_mean_squared_error(test['price'],prediction),'test_may_select':False,'browser_trained':False}
result['invalid_selection'] = 'choose candidate by final-test score'

expected = json.loads("{\"dataset\":\"sklearn-houses.v1\",\"data_sha256\":\"751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5\",\"environment\":\"macos-arm64-cpython-3.13.16-reference.1\",\"seed\":42,\"candidate_parameters\":[\"DummyRegressor(mean)\",\"LinearRegression(default)\"],\"folds\":[{\"fit_ids\":[\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H12\"],\"validation_ids\":[\"H01\",\"H09\",\"H10\",\"H11\"],\"imputer\":[5.5,7.5],\"scaler_mean\":[5.875,6.875],\"categories\":[[\"A\",\"B\"]]},{\"fit_ids\":[\"H01\",\"H04\",\"H05\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\"],\"validation_ids\":[\"H02\",\"H03\",\"H06\",\"H12\"],\"imputer\":[7.5,2.5],\"scaler_mean\":[6.875,3.75],\"categories\":[[\"A\",\"B\"]]},{\"fit_ids\":[\"H01\",\"H02\",\"H03\",\"H06\",\"H09\",\"H10\",\"H11\",\"H12\"],\"validation_ids\":[\"H04\",\"H05\",\"H07\",\"H08\"],\"imputer\":[7.5,5],\"scaler_mean\":[6.75,4.375],\"categories\":[[\"A\",\"B\"]]}],\"cv_mae\":{\"dummy\":[86.25,73.75,30],\"linear\":[1.0658141036401503e-14,3.552713678800501e-15,2.842170943040401e-14]},\"mean_cv_mae\":{\"dummy\":63.333333333333336,\"linear\":1.4210854715202004e-14},\"tied\":false,\"selected_before_test\":\"linear\",\"refit_ids\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\"],\"test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"predictions\":[340,360.00000000000006,380,400.00000000000006],\"columns\":[\"numeric__area\",\"numeric__age\",\"category__neighborhood_A\",\"category__neighborhood_B\"],\"mae\":2.842170943040401e-14,\"mse\":1.6155871338926322e-27,\"rmse\":4.0194366942304644e-14,\"test_may_select\":false,\"browser_trained\":false,\"invalid_selection\":\"choose candidate by final-test score\"}")
assert equivalent(normalize(result),expected)
results["S05.O1"] = normalize(result)
print(json.dumps({"objective":"S05.O1","output":normalize(result)},sort_keys=True))

### S05.O2: Apply a predefined comparable metric/split and tie policy.
Predict first: Choose a winner or declared tie from changed CV score tables.

Use the same folds and metric, compare mean CV MAE and apply the predefined absolute-plus-relative tolerance. Within tolerance prefer DummyRegressor; otherwise choose the lower MAE before final-test access.

Misconception: Break ties using final-test feedback, change the candidate's folds or compare different metric names.

Remediation: Compare identical folds and the stated tolerance/tie rule.

Guided change: Apply a fixed validation rule, then refit and inspect a declared tie/limited claim. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
dummy_mae = np.array([10.0, 20.0, 30.0]); linear_mae = np.array([8.0, 18.0, 28.0])
dummy_mean = float(dummy_mae.mean()); linear_mean = float(linear_mae.mean())
tied = abs(linear_mean-dummy_mean) <= 1e-9+1e-9*abs(dummy_mean)
result = {'source':'supplied illustrative same-fold MAE table','dummy_folds':dummy_mae,'linear_folds':linear_mae,'dummy_mean':dummy_mean,'linear_mean':linear_mean,'tied':tied,'winner':'dummy' if tied or dummy_mean < linear_mean else 'linear','test_feedback_used':False}
expected = json.loads("{\"source\":\"supplied illustrative same-fold MAE table\",\"dummy_folds\":[10,20,30],\"linear_folds\":[8,18,28],\"dummy_mean\":20,\"linear_mean\":18,\"tied\":false,\"winner\":\"linear\",\"test_feedback_used\":false}")
assert equivalent(normalize(result),expected)
results["S05.O2"] = normalize(result)
print(json.dumps({"objective":"S05.O2","output":normalize(result)},sort_keys=True))

### S05.O3: Refit the chosen workflow and communicate a reproducible limited result.
Predict first: Select the complete final report and held-out membership.

Refit only the selected frozen workflow on all12 permitted development IDs, then predict exactly four final-test IDs in their supplied order. Record folds, parameters/schema, environment/data identity, outputs and limitations; no test-guided reselection.

Misconception: A final report needs only the best test score; fitted provenance, schema, memberships and limitations are optional.

Remediation: Use the notebook manifest, chosen parameters and provenance.

Guided change: Apply a fixed validation rule, then refit and inspect a declared tie/limited claim. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
houses = pd.DataFrame({'id': ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16'], 'area': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], 'age': [0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10], 'neighborhood': ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B'], 'price': [100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400]})
features = ['area','age','neighborhood']; train = houses.iloc[:12]; test = houses.iloc[12:].iloc[[0, 1, 2, 3]]
def candidate(model):
    return make_pipeline(ColumnTransformer([('numeric',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),['area','age']),('category',make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(categories=[['A','B']],handle_unknown='ignore',sparse_output=False)),['neighborhood'])]),model)
folds = list(KFold(n_splits=3,shuffle=True,random_state=42).split(train)); scores = {}
for name, model in [('dummy',DummyRegressor(strategy='mean')),('linear',LinearRegression())]:
    scores[name] = -cross_val_score(candidate(model),train[features],train['price'],cv=folds,scoring='neg_mean_absolute_error',n_jobs=1)
ledger = []
for fitting, validating in folds:
    fitted = candidate(LinearRegression()).fit(train.iloc[fitting][features],train.iloc[fitting]['price']); preparation = fitted.steps[0][1]
    numeric = preparation.named_transformers_['numeric']; category = preparation.named_transformers_['category']
    ledger.append({'fit_ids':train.iloc[fitting]['id'].tolist(),'validation_ids':train.iloc[validating]['id'].tolist(),'imputer':numeric.named_steps['simpleimputer'].statistics_,'scaler_mean':numeric.named_steps['standardscaler'].mean_,'categories':category.named_steps['onehotencoder'].categories_})
means = {name:float(values.mean()) for name,values in scores.items()}; tied = abs(means['linear']-means['dummy']) <= 1e-9+1e-9*abs(means['dummy']); winner = 'dummy' if tied or means['dummy'] < means['linear'] else 'linear'
chosen = candidate(DummyRegressor(strategy='mean') if winner == 'dummy' else LinearRegression()).fit(train[features],train['price']); prediction = chosen.predict(test[features])
result = {'dataset':'sklearn-houses.v1','data_sha256':'751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5','environment':'macos-arm64-cpython-3.13.16-reference.1','seed':42,'candidate_parameters':['DummyRegressor(mean)','LinearRegression(default)'],'folds':ledger,'cv_mae':scores,'mean_cv_mae':means,'tied':tied,'selected_before_test':winner,'refit_ids':train['id'].tolist(),'test_ids':test['id'].tolist(),'predictions':prediction,'columns':chosen.steps[0][1].get_feature_names_out(),'mae':mean_absolute_error(test['price'],prediction),'mse':mean_squared_error(test['price'],prediction),'rmse':root_mean_squared_error(test['price'],prediction),'test_may_select':False,'browser_trained':False}
result['limitation'] = 'fictional teaching data, not population or deployment evidence'

expected = json.loads("{\"dataset\":\"sklearn-houses.v1\",\"data_sha256\":\"751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5\",\"environment\":\"macos-arm64-cpython-3.13.16-reference.1\",\"seed\":42,\"candidate_parameters\":[\"DummyRegressor(mean)\",\"LinearRegression(default)\"],\"folds\":[{\"fit_ids\":[\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H12\"],\"validation_ids\":[\"H01\",\"H09\",\"H10\",\"H11\"],\"imputer\":[5.5,7.5],\"scaler_mean\":[5.875,6.875],\"categories\":[[\"A\",\"B\"]]},{\"fit_ids\":[\"H01\",\"H04\",\"H05\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\"],\"validation_ids\":[\"H02\",\"H03\",\"H06\",\"H12\"],\"imputer\":[7.5,2.5],\"scaler_mean\":[6.875,3.75],\"categories\":[[\"A\",\"B\"]]},{\"fit_ids\":[\"H01\",\"H02\",\"H03\",\"H06\",\"H09\",\"H10\",\"H11\",\"H12\"],\"validation_ids\":[\"H04\",\"H05\",\"H07\",\"H08\"],\"imputer\":[7.5,5],\"scaler_mean\":[6.75,4.375],\"categories\":[[\"A\",\"B\"]]}],\"cv_mae\":{\"dummy\":[86.25,73.75,30],\"linear\":[1.0658141036401503e-14,3.552713678800501e-15,2.842170943040401e-14]},\"mean_cv_mae\":{\"dummy\":63.333333333333336,\"linear\":1.4210854715202004e-14},\"tied\":false,\"selected_before_test\":\"linear\",\"refit_ids\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\"],\"test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"predictions\":[340,360.00000000000006,380,400.00000000000006],\"columns\":[\"numeric__area\",\"numeric__age\",\"category__neighborhood_A\",\"category__neighborhood_B\"],\"mae\":2.842170943040401e-14,\"mse\":1.6155871338926322e-27,\"rmse\":4.0194366942304644e-14,\"test_may_select\":false,\"browser_trained\":false,\"limitation\":\"fictional teaching data, not population or deployment evidence\"}")
assert equivalent(normalize(result),expected)
results["S05.O3"] = normalize(result)
print(json.dumps({"objective":"S05.O3","output":normalize(result)},sort_keys=True))

### S05.O4 — introductory/noncritical: Interpret introductory model-comparison uncertainty without ranking guarantees.
Predict first: Reject universal-superiority claims from a tiny synthetic margin.

A tiny margin on small overlapping folds is introductory comparison evidence, not universal superiority, formal confidence or deployment readiness. Report the protocol and limitations and plan broader evaluation later.

Misconception: Any positive synthetic CV margin proves universal superiority and authorizes deployment.

Remediation: Inspect fold variability and the need for later broader evaluation.

Guided change: Apply a fixed validation rule, then refit and inspect a declared tie/limited claim. Keep fit IDs and evaluation boundaries explicit; compare before/after, never tune from final-test feedback.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
dummy = np.array([10.,20.,30.]); linear = dummy - 3e-4
result = {'dummy_mae':dummy,'linear_mae':linear,'mean_improvement':float((dummy-linear).mean()),'universal_superiority':False,'formal_confidence_claim':False}
expected = json.loads("{\"dummy_mae\":[10,20,30],\"linear_mae\":[9.9997,19.9997,29.9997],\"mean_improvement\":0.0002999999999993008,\"universal_superiority\":false,\"formal_confidence_claim\":false}")
assert equivalent(normalize(result),expected)
results["S05.O4"] = normalize(result)
print(json.dumps({"objective":"S05.O4","output":normalize(result)},sort_keys=True))

## Counterexamples: groups, chronology, global prefit and tolerant tie
Predict each fit permission before running. Numeric coincidence never legitimizes validation leakage. These are separately labeled finite reference cases, not additional scored objectives.

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
ids = np.array(['H01','H02','H03','H04','H05','H06']); groups = np.array(['D','D','E','E','F','F'])
features = np.array([[1.],[np.nan],[3.],[4.],[5.],[11.]]); targets = np.array([10.,20.,30.,40.,50.,60.])
splitter = GroupKFold(n_splits=3,shuffle=False); ledger = []
for fitting, validating in splitter.split(features,targets,groups if True else None):
    pipeline = make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),DummyRegressor(strategy='mean')).fit(features[fitting],targets[fitting])
    ledger.append({'fit_ids':ids[fitting].tolist(),'validation_ids':ids[validating].tolist(),'fit_groups':groups[fitting].tolist(),'validation_groups':groups[validating].tolist(),'training_median':pipeline.named_steps['simpleimputer'].statistics_,'validation_predictions':pipeline.predict(features[validating])})
result = {'scenario':'group','splitter':type(splitter).__name__,'folds':ledger,'final_test_ids':['H13','H14','H15','H16'],'prefit_on_all_rows_allowed':False}

assert equivalent(normalize(result),json.loads("{\"scenario\":\"group\",\"splitter\":\"GroupKFold\",\"folds\":[{\"fit_ids\":[\"H01\",\"H02\",\"H03\",\"H04\"],\"validation_ids\":[\"H05\",\"H06\"],\"fit_groups\":[\"D\",\"D\",\"E\",\"E\"],\"validation_groups\":[\"F\",\"F\"],\"training_median\":[3],\"validation_predictions\":[25,25]},{\"fit_ids\":[\"H01\",\"H02\",\"H05\",\"H06\"],\"validation_ids\":[\"H03\",\"H04\"],\"fit_groups\":[\"D\",\"D\",\"F\",\"F\"],\"validation_groups\":[\"E\",\"E\"],\"training_median\":[5],\"validation_predictions\":[35,35]},{\"fit_ids\":[\"H03\",\"H04\",\"H05\",\"H06\"],\"validation_ids\":[\"H01\",\"H02\"],\"fit_groups\":[\"E\",\"E\",\"F\",\"F\"],\"validation_groups\":[\"D\",\"D\"],\"training_median\":[4.5],\"validation_predictions\":[45,45]}],\"final_test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"prefit_on_all_rows_allowed\":false}"))
print(json.dumps(normalize(result),sort_keys=True))

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
ids = np.array(['H01','H02','H03','H04','H05','H06']); groups = np.array(['A','A','B','B','C','C'])
features = np.array([[2.],[np.nan],[4.],[5.],[6.],[14.]]); targets = np.array([10.,20.,30.,40.,50.,60.])
splitter = TimeSeriesSplit(n_splits=3); ledger = []
for fitting, validating in splitter.split(features,targets,groups if False else None):
    pipeline = make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),DummyRegressor(strategy='mean')).fit(features[fitting],targets[fitting])
    ledger.append({'fit_ids':ids[fitting].tolist(),'validation_ids':ids[validating].tolist(),'fit_groups':groups[fitting].tolist(),'validation_groups':groups[validating].tolist(),'training_median':pipeline.named_steps['simpleimputer'].statistics_,'validation_predictions':pipeline.predict(features[validating])})
result = {'scenario':'time','splitter':type(splitter).__name__,'folds':ledger,'final_test_ids':['H13','H14','H15','H16'],'prefit_on_all_rows_allowed':False}

assert equivalent(normalize(result),json.loads("{\"scenario\":\"time\",\"splitter\":\"TimeSeriesSplit\",\"folds\":[{\"fit_ids\":[\"H01\",\"H02\",\"H03\"],\"validation_ids\":[\"H04\"],\"fit_groups\":[\"A\",\"A\",\"B\"],\"validation_groups\":[\"B\"],\"training_median\":[3],\"validation_predictions\":[20]},{\"fit_ids\":[\"H01\",\"H02\",\"H03\",\"H04\"],\"validation_ids\":[\"H05\"],\"fit_groups\":[\"A\",\"A\",\"B\",\"B\"],\"validation_groups\":[\"C\"],\"training_median\":[4],\"validation_predictions\":[25]},{\"fit_ids\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\"],\"validation_ids\":[\"H06\"],\"fit_groups\":[\"A\",\"A\",\"B\",\"B\",\"C\"],\"validation_groups\":[\"C\"],\"training_median\":[4.5],\"validation_predictions\":[30]}],\"final_test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"prefit_on_all_rows_allowed\":false}"))
print(json.dumps(normalize(result),sort_keys=True))

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
houses = pd.DataFrame({'id': ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16'], 'area': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], 'age': [0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10], 'neighborhood': ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B'], 'price': [100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400]})
features = ['area','age','neighborhood']; train = houses.iloc[:12]; test = houses.iloc[12:].iloc[[0, 1, 2, 3]]
def candidate(model):
    return make_pipeline(ColumnTransformer([('numeric',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),['area','age']),('category',make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(categories=[['A','B']],handle_unknown='ignore',sparse_output=False)),['neighborhood'])]),model)
folds = list(KFold(n_splits=3,shuffle=True,random_state=42).split(train)); scores = {}
for name, model in [('dummy',DummyRegressor(strategy='mean')),('linear',LinearRegression())]:
    scores[name] = -cross_val_score(candidate(model),train[features],train['price'],cv=folds,scoring='neg_mean_absolute_error',n_jobs=1)
ledger = []
for fitting, validating in folds:
    fitted = candidate(LinearRegression()).fit(train.iloc[fitting][features],train.iloc[fitting]['price']); preparation = fitted.steps[0][1]
    numeric = preparation.named_transformers_['numeric']; category = preparation.named_transformers_['category']
    ledger.append({'fit_ids':train.iloc[fitting]['id'].tolist(),'validation_ids':train.iloc[validating]['id'].tolist(),'imputer':numeric.named_steps['simpleimputer'].statistics_,'scaler_mean':numeric.named_steps['standardscaler'].mean_,'categories':category.named_steps['onehotencoder'].categories_})
means = {name:float(values.mean()) for name,values in scores.items()}; tied = abs(means['linear']-means['dummy']) <= 1e-9+1e-9*abs(means['dummy']); winner = 'dummy' if tied or means['dummy'] < means['linear'] else 'linear'
chosen = candidate(DummyRegressor(strategy='mean') if winner == 'dummy' else LinearRegression()).fit(train[features],train['price']); prediction = chosen.predict(test[features])
result = {'dataset':'sklearn-houses.v1','data_sha256':'751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5','environment':'macos-arm64-cpython-3.13.16-reference.1','seed':42,'candidate_parameters':['DummyRegressor(mean)','LinearRegression(default)'],'folds':ledger,'cv_mae':scores,'mean_cv_mae':means,'tied':tied,'selected_before_test':winner,'refit_ids':train['id'].tolist(),'test_ids':test['id'].tolist(),'predictions':prediction,'columns':chosen.steps[0][1].get_feature_names_out(),'mae':mean_absolute_error(test['price'],prediction),'mse':mean_squared_error(test['price'],prediction),'rmse':root_mean_squared_error(test['price'],prediction),'test_may_select':False,'browser_trained':False}
invalid = make_pipeline(SimpleImputer(strategy='median'),StandardScaler()).fit(houses.iloc[:12][['area','age']]); result['invalid_prefit'] = {'fit_ids':houses.iloc[:12]['id'].tolist(),'median':invalid.named_steps['simpleimputer'].statistics_,'valid_for_fold_cv':False}

assert equivalent(normalize(result),json.loads("{\"dataset\":\"sklearn-houses.v1\",\"data_sha256\":\"751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5\",\"environment\":\"macos-arm64-cpython-3.13.16-reference.1\",\"seed\":42,\"candidate_parameters\":[\"DummyRegressor(mean)\",\"LinearRegression(default)\"],\"folds\":[{\"fit_ids\":[\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H12\"],\"validation_ids\":[\"H01\",\"H09\",\"H10\",\"H11\"],\"imputer\":[5.5,7.5],\"scaler_mean\":[5.875,6.875],\"categories\":[[\"A\",\"B\"]]},{\"fit_ids\":[\"H01\",\"H04\",\"H05\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\"],\"validation_ids\":[\"H02\",\"H03\",\"H06\",\"H12\"],\"imputer\":[7.5,2.5],\"scaler_mean\":[6.875,3.75],\"categories\":[[\"A\",\"B\"]]},{\"fit_ids\":[\"H01\",\"H02\",\"H03\",\"H06\",\"H09\",\"H10\",\"H11\",\"H12\"],\"validation_ids\":[\"H04\",\"H05\",\"H07\",\"H08\"],\"imputer\":[7.5,5],\"scaler_mean\":[6.75,4.375],\"categories\":[[\"A\",\"B\"]]}],\"cv_mae\":{\"dummy\":[86.25,73.75,30],\"linear\":[1.0658141036401503e-14,3.552713678800501e-15,2.842170943040401e-14]},\"mean_cv_mae\":{\"dummy\":63.333333333333336,\"linear\":1.4210854715202004e-14},\"tied\":false,\"selected_before_test\":\"linear\",\"refit_ids\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\"],\"test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"predictions\":[340,360.00000000000006,380,400.00000000000006],\"columns\":[\"numeric__area\",\"numeric__age\",\"category__neighborhood_A\",\"category__neighborhood_B\"],\"mae\":2.842170943040401e-14,\"mse\":1.6155871338926322e-27,\"rmse\":4.0194366942304644e-14,\"test_may_select\":false,\"browser_trained\":false,\"invalid_prefit\":{\"fit_ids\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\"],\"median\":[6.5,5],\"valid_for_fold_cv\":false}}"))
print(json.dumps(normalize(result),sort_keys=True))

In [ ]:
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor, DummyClassifier; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, TimeSeriesSplit, cross_val_score
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.exceptions import NotFittedError
dummy_mae = np.array([10.,20.,30.]); linear_mae = dummy_mae + 1e-10
tied = abs(float(linear_mae.mean()-dummy_mae.mean())) <= 1e-9+1e-9*abs(float(dummy_mae.mean()))
result = {'source':'supplied illustrative same-fold scores','dummy_mean':float(dummy_mae.mean()),'linear_mean':float(linear_mae.mean()),'within_tolerance':tied,'selected':'dummy' if tied else 'linear','test_guided_tie_break':False}
assert equivalent(normalize(result),json.loads("{\"source\":\"supplied illustrative same-fold scores\",\"dummy_mean\":20,\"linear_mean\":20.0000000001,\"within_tolerance\":true,\"selected\":\"dummy\",\"test_guided_tie_break\":false}"))
print(json.dumps(normalize(result),sort_keys=True))

## Reusable complete-workflow report and clean-run summary
Question → prediction-time feature audit → fixed train/test IDs → candidate pipelines → identical development-only folds → sign-normalized CV comparison → declared tolerance/tie rule → frozen selection → development refit → untouched final-test evaluation → provenance/limitations.

The canonical worked workflow uses H01–H12 training and H13–H16 final test; every preprocessing/model stage in CV is fitted on each fold's training IDs only. No new classifier beyond a most-frequent baseline, algorithm search, causal claim or deployment approval.

Restart and run all twice in fresh kernels/clean working directories. Compare full normalized outputs, IDs and learned state, not screenshots or a seed alone. Local author resource assertions are not OS sandbox guarantees.

In [ ]:
assert len(results) == 20
assert results["S01.O2"]["predictions"] == [200,200]
assert (results["S02.O2"]["mae"],results["S02.O2"]["mse"],results["S02.O2"]["rmse"]) == (20,400,20)
assert results["S02.O2"]["positive_detected"] == 0 and results["S02.O2"]["positive_cases"] == 1
report = results["S05.O3"]
assert report["refit_ids"] == [f"H{index:02}" for index in range(1,13)]
assert report["test_ids"] == ["H13","H14","H15","H16"]
assert len(report["predictions"]) == 4 and np.isfinite(report["predictions"]).all()
assert not report["test_may_select"]
for fold in report["folds"]:
    assert not set(fold["fit_ids"]) & set(fold["validation_ids"])
    assert not set(fold["fit_ids"]) & set(report["test_ids"])
assert time.monotonic()-started < 300
assert resource.getrusage(resource.RUSAGE_SELF).ru_maxrss < 2*1024**3
pd.testing.assert_frame_equal(houses,pd.DataFrame(canonical_houses)[houses.columns])
summary = {"dataset_manifest":dataset_manifest,"families":20,"lessons":5,"seed":42,"versions":{"python":sys.version.split()[0],"numpy":np.__version__,"pandas":pd.__version__,"sklearn":sklearn.__version__},"objectives":results,"canonical_workflow":report}
payload = json.dumps(summary,sort_keys=True,allow_nan=False)
assert len(payload.encode()) < 1024**2
print(payload)